# Lab 2: Who Wrote the Disputed Federalist Papers?

In this lab you will use the vectors from Lab 1 to guess who wrote the 12 disputed Federalist Papers, then test the same question with a second kind of vector built to capture writing style. The point is not the historical answer. It is to see that **what a vector represents decides what a vector database can find.**

The papers were published under the shared pen name "Publius." After Hamilton and Madison died, each was credited with lists of papers that conflicted on 12 of them: Nos. 49 to 58, 62, and 63. In 1964, statisticians Frederick Mosteller and David Wallace studied the rates of common "function words" such as *upon*, *whilst*, *by*, and *also*. Their analysis attributed all 12 to Madison, and most historians accept that conclusion.

You will need the `federalist_db` folder from Lab 1, in the same folder as this notebook. Everything else is reconstructed from the database itself.

| Part | Activity | Time |
| --- | --- | --- |
| 1 | Build one vector per paper | 10 min |
| 2 | Classify with semantic centroids | 20 min |
| 3 | Build and store style vectors | 20 min |
| 4 | Compare the methods | 15 min |

Run each cell in order with **Shift+Enter**.

## Part 1: From chunk vectors to paper vectors

Lab 1 stored one vector per paragraph chunk. To compare whole papers, average each paper's chunk vectors and rescale the result to length 1. Pull everything straight out of Chroma.

In [ ]:
import re
import numpy as np
import chromadb

client = chromadb.PersistentClient(path="./federalist_db")
collection = client.get_collection("federalist")

data = collection.get(include=["embeddings", "metadatas", "documents"])
emb   = np.array(data["embeddings"])
metas = data["metadatas"]
docs  = data["documents"]
print(emb.shape)                      # (1920, 384) if Lab 1 matched the reference run

In [ ]:
def unit(v):
    """Scale a vector to length 1."""
    return v / np.linalg.norm(v)

author = {m["paper"]: m["author"] for m in metas}
title  = {m["paper"]: m["title"] for m in metas}

paper_vecs = {}
for n in sorted(author):
    rows = [i for i, m in enumerate(metas) if m["paper"] == n]
    paper_vecs[n] = unit(emb[rows].mean(axis=0))

print(len(paper_vecs), "paper vectors")   # 85

Because every vector now has length 1, a plain dot product (`a @ b`) equals cosine similarity.

**Think about it.** Averaging is a lossy summary. What kinds of information about a paper survive averaging, and what kinds are blurred away?

## Part 2: Classify with semantic centroids

The simplest classifier averages all of Hamilton's paper vectors into one **centroid**, does the same for Madison, and assigns each disputed paper to whichever centroid is closer. Before trusting it on the disputed papers, you must test it on papers whose author you already know.

In [ ]:
HAM  = [n for n in paper_vecs if author[n] == "Hamilton"]
MAD  = [n for n in paper_vecs if author[n] == "Madison"]
DISP = [n for n in paper_vecs if author[n] == "Disputed"]
print(len(HAM), "Hamilton,", len(MAD), "Madison,", len(DISP), "disputed")

def centroid(vecs, papers):
    return unit(np.mean([vecs[n] for n in papers], axis=0))

def classify(vec, ham_c, mad_c):
    """Return the closer author and the margin (positive leans Madison)."""
    s_h, s_m = float(vec @ ham_c), float(vec @ mad_c)
    return ("Madison" if s_m > s_h else "Hamilton"), s_m - s_h

# Every method's key measures are saved here and used to fill in the Part 4 table.
results = {}

### Step 1: validate with leave-one-out

Hide one known paper, build centroids from the rest, and see whether the hidden paper is classified correctly. Repeat for every known paper.

Report accuracy separately for each author. With 51 Hamilton papers and only 14 Madison papers, overall accuracy alone could hide a classifier that is bad at recognizing Madison.

In [ ]:
def leave_one_out(vecs):
    """Return {author: (correct, total)} and print it."""
    counts = {"Hamilton": [0, 0], "Madison": [0, 0]}
    for n in HAM + MAD:
        ham_c = centroid(vecs, [p for p in HAM if p != n])
        mad_c = centroid(vecs, [p for p in MAD if p != n])
        pred, _ = classify(vecs[n], ham_c, mad_c)
        counts[author[n]][0] += pred == author[n]
        counts[author[n]][1] += 1
    for a, (right, total) in counts.items():
        print(f"{a}: {right}/{total} correct")
    return {a: tuple(c) for a, c in counts.items()}

results["Semantic"] = {"loo": leave_one_out(paper_vecs)}

### Step 2: classify the disputed papers

A positive margin leans Madison; a negative one leans Hamilton. Margins near zero mean the classifier is unsure.

In [ ]:
def predict_disputed(vecs):
    """Return {paper: (predicted author, margin)} and print it."""
    ham_c, mad_c = centroid(vecs, HAM), centroid(vecs, MAD)
    preds = {}
    for n in DISP:
        pred, margin = classify(vecs[n], ham_c, mad_c)
        preds[n] = (pred, margin)
        print(f"No. {n}: {pred:8s} (margin {margin:+.3f})")
    return preds

results["Semantic"]["disputed"] = predict_disputed(paper_vecs)

### Step 3: look at the neighbors

For each disputed paper, list its three most similar known papers, with their titles.

In [ ]:
def nearest_known(vecs, n, k=3):
    sims = sorted(((float(vecs[n] @ vecs[p]), p) for p in HAM + MAD), reverse=True)[:k]
    return [(p, author[p], round(s, 3)) for s, p in sims]

for n in DISP:
    print(f"No. {n}: {title[n]}")
    for p, a, s in nearest_known(paper_vecs, n):
        print(f"    {s:.3f}  No. {p} ({a}): {title[p]}")

**Investigate.** Are the neighbors similar because they share an *author* or because they share a *topic*? Madison's Nos. 47 and 48 open a discussion of the separation of powers that the disputed Nos. 49 to 51 continue, so topic and author are tangled together. If the classifier says "Madison," is that evidence about authorship?

## Part 3: Style vectors from function words

Now build a different vector for each paper, in the spirit of Mosteller and Wallace: how often the author uses 20 common function words, per 1,000 words. These words carry almost no topic, so they mostly reflect habit. A vector database does not care where vectors come from, so you will store these in a second Chroma collection.

### Step 1: compute the rates

Rebuild each paper's text from the chunks already in the database.

In [ ]:
FUNCTION_WORDS = ["upon", "whilst", "while", "by", "also", "an", "there", "on",
                  "enough", "although", "both", "to", "of", "this", "would",
                  "may", "must", "which", "though", "always"]

paper_text = {}
for doc, m in zip(docs, metas):
    paper_text.setdefault(m["paper"], []).append(doc)

def rates(text):
    tokens = re.findall(r"[a-z]+", text.lower())
    return np.array([1000 * tokens.count(w) / len(tokens) for w in FUNCTION_WORDS])

raw_rates = {n: rates(" ".join(chunks)) for n, chunks in paper_text.items()}

print(f"{'word':10s} {'Hamilton':>9s} {'Madison':>9s}   (per 1,000 words)")
for w, h, m in zip(FUNCTION_WORDS,
                   np.mean([raw_rates[n] for n in HAM], axis=0),
                   np.mean([raw_rates[n] for n in MAD], axis=0)):
    print(f"{w:10s} {h:9.2f} {m:9.2f}")

Which words differ most between the two authors? Look especially at *upon*, *while*, and *whilst*.

### Step 2: standardize

Common words like *of* and *to* occur dozens of times per 1,000 words, while *whilst* occurs well under once. Without rescaling, the common words would dominate every distance. Convert each rate to a z-score using the known papers, then scale to length 1.

In [ ]:
known = np.array([raw_rates[n] for n in HAM + MAD])
mu, sd = known.mean(axis=0), known.std(axis=0) + 1e-9    # avoid dividing by zero
style_vecs = {n: unit((r - mu) / sd) for n, r in raw_rates.items()}
print(len(style_vecs), "style vectors of", len(FUNCTION_WORDS), "dimensions")

### Step 3: store them in Chroma

This time you supply the embeddings yourself instead of letting Chroma call a model.

In [ ]:
# Start fresh if you re-run this cell. Check explicitly rather than
# catching errors, so any real problem with the delete is reported.
existing = [c.name for c in client.list_collections()]
if "federalist_style" in existing:
    client.delete_collection("federalist_style")

style = client.create_collection("federalist_style",
                                 configuration={"hnsw": {"space": "cosine"}})
style.add(ids=[f"fed{n:02d}" for n in style_vecs],
          embeddings=[v.tolist() for v in style_vecs.values()],
          metadatas=[{"paper": n, "author": author[n]} for n in style_vecs])
print(style.count())     # 85 vectors of 20 dimensions each

### Step 4: classify

First run the same centroid tests from Part 2 on the new vectors.

In [ ]:
results["Style"] = {"loo": leave_one_out(style_vecs)}

In [ ]:
results["Style"]["disputed"] = predict_disputed(style_vecs)

Then let the database do the work with a nearest-neighbor vote: for each disputed paper, query the `K` most stylistically similar *known* papers and count their authors. A paper gets a Madison majority when at least `MAJORITY` of its neighbors are Madison papers.

Keep in mind that there are 51 Hamilton papers and only 14 Madison papers. If the neighbors were effectively random, you would expect about `K` × 14/65 Madison votes per paper by chance (about 1.1 when `K` is 5), so judge the vote counts against that baseline, not just against the majority line.

Note the metadata filter: without it, a disputed paper's nearest neighbors could be other disputed papers, which can't vote.

In [ ]:
K = 5                    # number of neighbors that vote; try 3, 7, or 9
MAJORITY = K // 2 + 1    # votes needed for a majority (3 when K = 5)

chance = K * len(MAD) / (len(HAM) + len(MAD))
print(f"K = {K}: a majority is {MAJORITY} votes; about {chance:.1f} Madison votes expected by chance\n")

knn_votes = {}
for n in DISP:
    res = style.query(query_embeddings=[style_vecs[n].tolist()], n_results=K,
                      where={"author": {"$in": ["Hamilton", "Madison"]}})
    votes = [m["author"] for m in res["metadatas"][0]]
    knn_votes[n] = votes.count("Madison")          # Madison votes out of K
    print(f"No. {n}: Madison {votes.count('Madison')}, Hamilton {votes.count('Hamilton')}")

results["Style"]["knn"] = knn_votes
results["Style"]["knn_settings"] = (K, MAJORITY)

## Part 4: Compare the two methods

The cell below builds the comparison table from the measures saved in `results`. Compare it with Mosteller and Wallace's verdict: Madison for all 12.

A margin counts as "near zero" when its size is below `NEAR_ZERO`. The two kinds of vectors produce margins on different scales, so look at the actual margins in Parts 2 and 3 and try a few thresholds.

In [ ]:
from IPython.display import Markdown, display

NEAR_ZERO = 0.01     # margins smaller than this count as "unsure"

def summarize(r):
    """Turn one method's saved results into the summary measures."""
    h_right, h_total = r["loo"]["Hamilton"]
    m_right, m_total = r["loo"]["Madison"]
    d = r["disputed"]
    row = {
        f"Leave-one-out: Hamilton correct (of {h_total})": h_right,
        f"Leave-one-out: Madison correct (of {m_total})": m_right,
        f"Disputed papers assigned to Madison (of {len(d)})":
            sum(pred == "Madison" for pred, _ in d.values()),
        f"Disputed papers with margin near zero (< {NEAR_ZERO})":
            sum(abs(margin) < NEAR_ZERO for _, margin in d.values()),
    }
    if "knn" in r:
        k, majority = r["knn_settings"]
        row[f"k-NN vote (K = {k}): disputed papers with at least {majority} Madison votes (of {len(d)})"] = \
            sum(v >= majority for v in r["knn"].values())
    return row

def show_comparison():
    summaries = {name: summarize(r) for name, r in results.items()}
    measures = list(dict.fromkeys(m for s in summaries.values() for m in s))
    lines = ["| Measure | " + " | ".join(summaries) + " |",
             "| --- | " + " | ".join("---" for _ in summaries) + " |"]
    for m in measures:
        lines.append(f"| {m} | " + " | ".join(str(s.get(m, "—")) for s in summaries.values()) + " |")
    display(Markdown("\n".join(lines)))
    return summaries

summary = show_comparison()

The paper-by-paper detail is often more revealing than the totals. Margins are positive when a method leans Madison.

In [ ]:
def show_disputed_detail():
    names = list(results)
    header = ["Paper"] + [f"{name} margin" for name in names]
    if "knn" in results.get("Style", {}):
        header.append(f"k-NN Madison votes (of {results['Style']['knn_settings'][0]})")
    lines = ["| " + " | ".join(header) + " |", "| " + " | ".join("---" for _ in header) + " |"]
    for n in DISP:
        cells = [f"No. {n}"]
        for name in names:
            pred, margin = results[name]["disputed"][n]
            cells.append(f"{margin:+.3f} ({pred})")
        if "knn" in results.get("Style", {}):
            cells.append(str(results["Style"]["knn"][n]))
        lines.append("| " + " | ".join(cells) + " |")
    display(Markdown("\n".join(lines)))

show_disputed_detail()

Two outcomes are worth recognizing:

- **A method can get the "right" answer for the wrong reason.** If the semantic classifier says Madison but fails leave-one-out validation, or its neighbors are clearly topic matches, its answer is not evidence of authorship.
- **A method can be useful but not decisive.** Twenty word rates and a simple centroid are far cruder than Mosteller and Wallace's Bayesian model. Results that lean the right way with narrow margins are an honest outcome, not a failure.

### Extension (optional)

How much of the style signal came from the single word *upon*? The cell below rebuilds the style vectors without it, saves the results as a third method, and redraws both Part 4 tables with a new column.

In [ ]:
reduced = [w for w in FUNCTION_WORDS if w != "upon"]
idx = [FUNCTION_WORDS.index(w) for w in reduced]

known_r = known[:, idx]
mu_r, sd_r = known_r.mean(axis=0), known_r.std(axis=0) + 1e-9
style_no_upon = {n: unit((r[idx] - mu_r) / sd_r) for n, r in raw_rates.items()}

results["Style without upon"] = {"loo": leave_one_out(style_no_upon),
                                 "disputed": predict_disputed(style_no_upon)}

# Rebuild both tables with the new column added
summary = show_comparison()
show_disputed_detail()

## Deliverables

Submit this notebook (with its outputs, including the Part 4 tables) and a one- to two-page write-up answering the questions below.

### Discussion questions

1. Which vectors performed better on leave-one-out validation? Why does validation on known papers matter before you trust any prediction on the disputed ones?
2. For two disputed papers, list their nearest semantic neighbors and their titles. Do the neighbors share an author, a topic, or both?
3. The same database, the same query code, and the same distance metric gave different answers depending only on how the vectors were built. Who is responsible for the quality of a vector search: the database, or the person who chose the representation?
4. Suppose you were building a plagiarism detector for student essays. Would you use semantic embeddings, style vectors, or both? Explain.
5. The style collection holds 85 vectors of 20 dimensions; the semantic collection holds thousands of 384-dimensional vectors. At which scale does an approximate index like HNSW start to matter, and how would you find out?